<style>
.proposal-title {background: linear-gradient(120deg,#17324d,#315d70); color:white; padding:34px 38px; border-radius:12px; margin-bottom:20px;}
.proposal-title h1 {margin:0 0 10px 0; font-size:34px; line-height:1.15;}
.proposal-title p {margin:5px 0; font-size:16px;}
.key-box {background:#eef5f2; border-left:5px solid #2f766b; padding:12px 16px; margin:14px 0;}
.caution-box {background:#fff7e8; border-left:5px solid #c88422; padding:12px 16px; margin:14px 0;}
.caption {color:#4f5b62; font-size:0.92em; text-align:center; margin-top:4px;}
table {font-size:0.94em;}
th {background:#e7eef2;}
</style>

<div class="proposal-title">
<h1>Dynamic Duration-Dependent HMM-WGAN for Multivariate Financial Return Simulation and Risk Measurement</h1>
<p><strong>Project Proposal</strong></p>
<p>Truong Co Nguyen · Nam Hoang · Trung Hieu Ha</p>
<p>WorldQuant University · 19 July 2026</p>
</div>

**Proposal purpose.** This document defines the research problem, methodology, preliminary Phase 1 evidence, project risks, and delivery plan for a dynamic extension of the fixed-transition HMM-WGAN framework. It is intentionally decision-oriented: each diagnostic is tied to a modeling choice, and each preliminary result is separated from work planned for subsequent milestones.

**Contents**

1. Executive Summary  
2. Problem, Research Question, and Objectives  
3. Literature and Competitor Positioning  
4. Data and Research Design  
5. Phase 1 HMM Diagnostics  
6. Dynamic Transition Models and Selection  
7. Preliminary Backtest, Feature Importance, and Duration Effects  
8. Pain Points, Obstacles, and Mitigation  
9. Milestone Plan and Success Criteria  
10. Works Cited


# 1. Executive Summary

Financial return distributions are non-Gaussian, dependent across assets, and unstable through time. A Hidden Markov Model (HMM) can organize this instability into latent market regimes, while regime-specific Wasserstein Generative Adversarial Networks (WGANs) can learn flexible multivariate return distributions. The closest benchmark, the HMM-WGAN architecture of Istiaque, Pun, and Yong, nevertheless propagates regimes through a fixed first-order transition matrix. That transition mechanism is memoryless and time-homogeneous: conditional on today’s regime, it cannot respond to market indicators or to how long the market has occupied that regime.

This project proposes a **dynamic, covariate- and duration-aware transition layer**. Phase 1 calibrates an HMM and compares four one-step regime-probability models: the fixed transition matrix, multinomial logistic regression (MLR), regularized multinomial logistic GAM (MLG), and a group-weighted MLG that can penalize duration features more strongly than market features. Later phases will insert the selected transition mechanism into a rolling HMM-WGAN simulation and compare portfolio risk estimates with the fixed-transition benchmark. The proposal develops the research direction established in the team’s prior problem statement and literature review (Nguyen, Hoang, and Ha, “Problem Statement”; Nguyen, Hoang, and Ha, “Literature Review”).

<div class="key-box"><strong>Preliminary decision.</strong> Four regimes have the strongest BIC, ICL, and bootstrap likelihood-ratio support, but three regimes produce the best validation log likelihood. We therefore use q4 as the richer primary specification and retain q3 as a robustness benchmark. For q4 transition prediction, the original MLG has the best out-of-sample log loss (0.4075) and Brier score (0.2050), improving on the fixed matrix (0.4391 and 0.2083). Group weighting materially rebalances feature importance in q2, but q4 validation selects no extra duration penalty; this prevents us from claiming that duration dominance is universally removable by stronger regularization.</div>

The diagnostics also qualify the result. Gaussian emission assumptions are frequently rejected, volatility clustering remains in standardized residuals, regime parameters and transitions are not fully stable, and rare transitions produce wide partial-effect uncertainty bands. These are not reasons to abandon the architecture. Instead, they motivate the WGAN emission layer, rolling re-estimation, probability-based rather than hard-label evaluation, and careful uncertainty reporting.


# 2. Problem, Research Question, and Objectives

## 2.1 Problem statement

Risk engines must simulate joint asset returns through benign and stressed conditions. Standard multivariate GARCH models capture conditional volatility and correlation, but their distributional restrictions can miss nonlinear dependence and crisis tails. Regime-switching models allow distinct market states, yet a conventional homogeneous HMM imposes

$$P(S_{t+1}=j\mid S_t=i,\mathcal{F}_t)=a_{ij},$$

where the transition probability is constant through time and independent of market covariates and regime age. This is restrictive when transition risk changes with credit, equity, commodity, or bond indicators, or when the probability of leaving a state depends on its elapsed duration. A fixed matrix may consequently generate regime paths with unrealistic timing even when its unconditional transition frequencies are well estimated.

The proposed architecture replaces $a_{ij}$ with a learned probability vector

$$\boldsymbol{p}_{t+1}=g(S_t,S_{t-1:t-L_s},X_t,X_{t-1:t-L_x},D_t),$$

where $X_t$ denotes market indices and $D_t$ contains global and state-specific duration information. The resulting regime path then determines which regime-specific WGAN generates the next multivariate return vector.

## 2.2 Research question

> **Does replacing the fixed HMM transition matrix with a dynamic, covariate- and duration-aware multinomial transition model improve the realism of simulated multivariate returns and the accuracy of portfolio risk estimates?**

Supporting questions are:

1. How many HMM regimes are statistically defensible, stable enough to interpret, and useful out of sample?
2. Do nonlinear MLG transition functions improve proper probability scores relative to fixed transitions and linear MLR?
3. Does duration contain incremental transition information, and can group-weighted regularization prevent it from crowding out economically relevant market indices?
4. When embedded in the HMM-WGAN, does the dynamic transition layer improve stylized facts, dependence, VaR, CVaR, and exceedance behavior?

## 2.3 Objectives and testable deliverables

| Objective | Phase 1 evidence / later deliverable | Decision criterion |
|---|---|---|
| Calibrate and diagnose 2-, 3-, and 4-regime HMMs | GOF, bootstrap count tests, occupancy, covariance, emission, residual, duration, and stability diagnostics | Prefer parsimonious count supported by in-sample, holdout, and stability evidence |
| Build dynamic transition models | Independently selected MLR, regularized MLG, and group-weighted MLG | Validation log loss first; Brier score, significance, complexity, and stability as supporting criteria |
| Quantify economic drivers | Coefficient-group importance, conditional permutation tests, duration-hazard and partial-effect plots | Distinguish genuine predictive contribution from scale- or basis-dependent coefficient magnitude |
| Benchmark probability forecasts | 2017–2023 fixed versus dynamic out-of-sample comparison | Lower log loss and multiclass Brier score without unacceptable instability |
| Evaluate full HMM-WGAN | M5 rolling simulation and portfolio backtests | Better stylized-fact replication and risk forecasts under prespecified metrics |


# 3. Literature and Competitor Positioning

The project combines four established lines of work rather than treating a single model as sufficient.

**Stylized facts and conditional dependence.** Financial returns exhibit heavy tails, volatility clustering, aggregation effects, and nonlinear dependence (Cont). GARCH and DCC models remain important conditional benchmarks (Bollerslev; Engle, “Dynamic Conditional Correlation”), but a fully parametric law can underrepresent complex multivariate tails.

**Regime switching.** Hamilton formalizes latent-state changes in economic time series, and Rabiner provides the foundational HMM machinery. Ang and Timmermann and Guidolin show why regimes are economically meaningful in finance. However, standard HMM transitions are constant. Filardo introduces time-varying transition probabilities, while Durland and McCurdy and Maheu and McCurdy show that business-cycle and equity-return dynamics can depend on regime duration. Hidden semi-Markov research further establishes duration as a direct state-process component (Yu; Giner and Zakamulin).

**Flexible generative distributions.** GANs learn distributions without prescribing a Gaussian likelihood (Goodfellow et al.). Wasserstein distance and gradient penalties improve training behavior (Arjovsky, Chintala, and Bottou; Gulrajani et al.). Financial applications demonstrate potential for time-series and scenario generation (Takahashi, Chen, and Tanaka-Ishii; Wiese et al.; Masi et al.).

**Competitor and gap.** Istiaque, Pun, and Yong combine HMM regimes with regime-specific WGANs and are therefore the closest competitor. Their architecture supplies flexible conditional emissions but retains fixed HMM regime transitions. Our incremental contribution is a directly comparable dynamic transition layer, estimated without altering the benchmark HMM or the existing fixed and MLR outputs.

| Approach | Regimes | Flexible multivariate distribution | Covariate-dependent transitions | Explicit duration | Interpretable transition effects |
|---|---:|---:|---:|---:|---:|
| GARCH / DCC | No | Limited by parametric law | N/A | No | Moderate |
| Fixed-transition HMM | Yes | Usually parametric | No | Geometric only | High |
| Fixed HMM-WGAN competitor | Yes | Yes | No | No | Moderate |
| Proposed dynamic HMM-WGAN | Yes | Yes | Yes | Yes | High through MLR/MLG effects |

<div class="key-box"><strong>Research contribution.</strong> The novelty is not merely adding another classifier. It is connecting an interpretable, duration-aware regime law to a flexible multivariate generator and evaluating the incremental value with probability scores and portfolio-risk tests under a common rolling design.</div>


# 4. Data and Research Design

## 4.1 Temporal design

Phase 1 uses a strict chronological split to avoid look-ahead bias:

| Segment | Period | Role |
|---|---|---|
| Training | 3 Jan. 2000 to 1 Jun. 2009 | HMM fitting, scaling, transition-model estimation, significance screening |
| Validation | 2 Jun. 2009 to 31 Dec. 2016 | Hyperparameter and architecture selection |
| Prediction / test | 3 Jan. 2017 to 5 Dec. 2023 | Untouched out-of-sample probability comparison |

The common HMM diagnostic feature set is SPXT, DBLCIX, IBOXIG, JPEICORE, LT11TRUU, and LBUTTRUU. These series span equity, commodity, credit, inflation-sensitive, and bond-market dimensions. Standardization parameters are learned only from training data. The same six-dimensional diagnostic input is used for q2, q3, and q4 so regime-count comparisons are not confounded by different information sets.

## 4.2 End-to-end architecture

1. Fit q2, q3, and q4 Gaussian HMMs with 50 starts and retain the highest-likelihood converged solution for each count.
2. Diagnose regime count, numerical conditioning, distributional assumptions, duration behavior, residual dependence, and temporal stability.
3. Infer the latent regime sequence and construct lagged-state, market, and duration predictors.
4. Independently select MLR, original regularized MLG, and group-weighted MLG using training and validation data only.
5. Freeze each selected model and compare one-step probabilities on 2017–2023 data against the fixed HMM transition matrix.
6. In M5, use each transition law to simulate regime paths and route observations to regime-specific WGAN generators.
7. Compare simulated return distributions and portfolio risks with identical assets, windows, random seeds, and evaluation metrics.

**Reproducibility principle.** Model-search tables, fitted probabilities, diagnostics, and figures are written to the Phase 1 output directory. Fixed-transition and original MLR calibrations are not overwritten when the MLG variants are added, preserving a clean experimental control.


# 5. Phase 1 HMM Diagnostics

## 5.1 Regime count and goodness of fit

No single statistic settles the number of regimes. We therefore triangulate among fit, separation, out-of-sample generalization, occupancy, and a simulation-based nested comparison.

- **BIC** rewards likelihood but penalizes parameter count, addressing overfitting as q increases (Schwarz). Lower is preferred.
- **ICL** adds a classification-uncertainty penalty to BIC, so it favors both fit and clearly separated regimes (Biernacki, Celeux, and Govaert). Lower is preferred.
- **Validation and prediction log likelihood per observation** ask whether the fitted emission/state process generalizes temporally. Higher, meaning less negative, is preferred.
- **Minimum smoothed occupancy** detects nearly empty states that may be numerical artifacts or too data-poor for later WGAN training.
- **Parametric-bootstrap likelihood-ratio tests** compare q with q+1 regimes. The ordinary chi-square reference is unreliable because mixture/HMM parameters are unidentified under the smaller model; bootstrap simulation supplies an empirical null distribution (McLachlan). We use 100 replications, so p-values should be treated as preliminary rather than ultra-precise.

| Regimes | BIC | ICL | Validation LL / row | Test LL / row | Minimum occupancy | Unstable regimes |
|---:|---:|---:|---:|---:|---:|---:|
| 2 | 30,369.68 | 30,785.26 | -6.5884 | -6.4527 | 21.66% | 2 / 2 |
| 3 | 29,366.09 | 29,907.65 | **-6.5508** | **-6.3803** | 9.66% | 3 / 3 |
| 4 | **28,936.16** | **29,373.83** | -6.6007 | -6.4611 | 11.13% | 3 / 4 |

| Bootstrap comparison | LR statistic | Bootstrap p-value | 5% decision |
|---|---:|---:|---|
| q2 versus q3 | 1,251.54 | 0.0099 | Reject q2 |
| q3 versus q4 | 693.38 | 0.0396 | Reject q3 |

<img src="phase_1/hmm_diagnostics_model_selection.png" alt="HMM model-selection diagnostics for two, three, and four regimes" width="900">
<p class="caption"><strong>Figure 1.</strong> HMM regime-count, fit, occupancy, and bootstrap evidence. Source: Phase 1 diagnostics.</p>

<div class="caution-box"><strong>Inference.</strong> BIC, ICL, and both bootstrap tests support q4, while q3 is better on both holdout likelihood measures. This disagreement is substantively useful: q4 extracts richer crisis/market structure, but its extra state does not improve the Gaussian HMM predictive likelihood. However, the later is not important as we do not use HMM component to simulate returns.</div>


## 5.2 Assumption and stability checks

A high-likelihood HMM can still be unsuitable if covariance estimates are ill-conditioned, Gaussian emissions are badly misspecified, residual dependence remains, durations contradict the memoryless assumption, or regimes drift across subsamples. The following diagnostics test those failure modes.

| Diagnostic | Why it is required | Preliminary result | Implication |
|---|---|---|---|
| Covariance eigenvalues and condition number | Near-singular covariance matrices can create artificial likelihood gains and unstable state assignment. The condition number compares largest with smallest eigenvalue. | All covariances are positive definite; maximum condition number is 89.12, far below the prespecified $10^8$ warning threshold. | Numerical degeneracy does not explain q4’s fit advantage. This is an important stability pass. |
| Jarque–Bera emission normality with Bonferroni control | Gaussian HMM emissions assume within-state skewness and kurtosis are consistent with normality. Multiple-test adjustment avoids declaring failure merely because many state-feature pairs are examined (Jarque and Bera). | Rejections: q2 12/12, q3 12/18, q4 14/24 state-feature pairs. | Gaussian emissions organize regimes but do not fully model within-regime tails. This directly motivates regime-specific WGANs. |
| Ljung–Box residual autocorrelation | Standardized residuals should not retain material linear predictability if the fitted dynamics are adequate (Ljung and Box). | At lag 20, 2 of 6 features reject no autocorrelation for each model and holdout split. | Some serial structure remains; rolling estimation and lag sensitivity are needed. |
| ARCH-LM residual test | Squared-residual dependence detects omitted volatility clustering (Engle, “Autoregressive Conditional Heteroscedasticity”). | All 6 features reject no ARCH effects in validation and prediction for q2, q3, and q4. | Regimes alone do not absorb conditional heteroskedasticity. The generative layer must reproduce clustered volatility, and a GARCH-type sensitivity benchmark is warranted. |
| Geometric-duration test | A first-order homogeneous HMM implies a constant exit hazard and hence geometric state duration. Rejecting that law supports explicit duration dependence. | q2 rejects 2/2 states; q3 rejects 0/3; q4 rejects 1/4 at 5% (state 1), with state 0 borderline at p=0.0519. | Duration misspecification is count- and state-specific, not universal. Duration belongs in the transition experiment but should be regularized rather than assumed dominant. |
| Segment stability of emissions and transitions | A regime useful for risk simulation should retain interpretable parameters through time; structural breaks make one full-sample matrix misleading. | Unstable regimes: q2 2/2, q3 3/3, q4 3/4. Fixed transition frequencies also vary across training segments. | Use rolling/expanding estimation and report sensitivity. q4 state 3 is the only state passing all implemented stability screens. |

<img src="phase_1/hmm_diagnostics_assumption_checks.png" alt="HMM assumption and stability diagnostic summary" width="900">
<p class="caption"><strong>Figure 2.</strong> Numerical, distributional, duration, residual, and stability checks. Source: Phase 1 diagnostics.</p>

**Economic reading of q4 states.** Average return profiles identify state 0 as the most benign state: equities and commodities are positive and credit is supportive. State 3 is the most severely stressed state: equity, commodity, and emerging-credit returns are sharply negative while nominal bonds are positive. These labels are assigned after estimation from economic characteristics; the HMM’s numeric state IDs have no intrinsic ordering.

<div class="key-box"><strong>Overall HMM conclusion.</strong> q4 is numerically sound and statistically supported as a richer partition, but Gaussianity, volatility, and stationarity assumptions are not fully satisfied. The correct inference is therefore “useful latent-state scaffold with documented misspecification,” not “fully adequate return model.”</div>


# 6. Dynamic Transition Models and Selection

Let $S_t\in\{0,\ldots,K-1\}$ be the current regime and $Z_t$ the information vector containing categorical state lags, current/lagged market indices, global duration, and regime-specific durations. Every model outputs a normalized K-vector, so $\sum_{k=0}^{K-1}p_{t,k}=1$ at each prediction date.

## 6.1 Multinomial logistic regression

$$p_{t,k}=\frac{\exp(\eta_{t,k})}{\sum_{j=0}^{K-1}\exp(\eta_{t,j})},\qquad \eta_{t,k}=\alpha_k+Z_t^{\mathsf T}\beta_k.$$

The MLR is a standardized, L2-regularized linear-log-odds benchmark. Candidate state and exogenous lag depths are each 0, 1, or 2. Models are fitted on training data and ranked on validation log loss, followed by Brier score and lower lag complexity. Its strength is interpretability; its weakness is that one coefficient forces a constant marginal slope across the full covariate range.

## 6.2 Regularized multinomial logistic GAM

$$\eta_{t,k}=\alpha_k+\gamma_k(S_t,S_{t-1:t-L_s})+\sum_{m=1}^{M}f_{k,m}(Z_{t,m}).$$

The MLG retains identifiable categorical state effects while replacing continuous linear terms with cubic B-splines. Duration is transformed as $\log(1+D)$ before spline expansion. A second-difference P-spline penalty discourages rough curves, and a small ridge floor stabilizes sparse multinomial classes (Eilers and Marx; Hastie and Tibshirani). The independent search covers state lags 0–2, exogenous lags 0–2, 4 or 6 knots, and regularization strengths $C\in\{0.01,0.1,1,10\}$.

Term screening uses training data only. Stable designs use joint Wald tests; sparse or ill-conditioned designs use a regularized parametric bootstrap. Terms must satisfy the 5% significance threshold before validation ranking. Validation selection then prioritizes log loss, Brier score, accuracy, retained-term count, and a fixed complexity tie-break. This order limits both data snooping and the tendency of a flexible spline basis to fit unsupported curvature.

## 6.3 Group-weighted multinomial logistic GAM

The weighted model keeps the original MLG architecture intact and refits an additional model with group-specific regularization:

$$\mathcal{L}(\theta)=-\ell(\theta)+\frac{1}{2C}\left[w_s\lVert\beta_s\rVert_2^2+w_m\lVert\beta_m\rVert_2^2+w_d\lVert\beta_d\rVert_2^2\right]+\text{P-spline roughness penalty}.$$

Here $w_s=1$, market weights are searched over $w_m\in\{0.5,1\}$, and duration weights over $w_d\in\{1,2,4,8\}$. A larger weight causes more shrinkage. The model reuses only training-qualified architectures and independently selects weights and $C$ on validation performance. This is a controlled response to duration dominance, not an arbitrary manual suppression of duration.

**Selection metrics.** Multiclass log loss is the primary criterion because it strongly penalizes confident errors; the Brier score measures squared probability-vector error (Brier). Accuracy is secondary because persistent regimes create a dominant “stay” class: a model can be accurate while assigning poor probabilities to the transitions that matter for risk.


## 6.4 Selected model specifications

| Regimes | Model | Selected state lags | Selected exogenous lags | Spline knots | C | Group weights $(w_s,w_m,w_d)$ | Retained specification |
|---:|---|---:|---:|---:|---:|---|---|
| 2 | MLR | 1 | 0 | — | 1.00 | — | State terms, current market variables, global and state durations |
| 2 | MLG | 0 | 2 | 6 | 0.01 | — | 21 terms: current state; six market series at lags 0–2; state 0 and state 1 durations |
| 2 | Group-weighted MLG | 0 | 2 | 6 | 0.10 | (1, 0.5, 8) | Same significant architecture as original q2 MLG, refitted with stronger duration shrinkage |
| 4 | MLR | 0 | 0 | — | 1.00 | — | Current state, six current market variables, global and state durations |
| 4 | MLG | 0 | 0 | 6 | 0.01 | — | 11 terms: current state, six current market series, four state-specific durations |
| 4 | Group-weighted MLG | 0 | 0 | 6 | 0.01 | (1, 0.5, 1) | Same q4 architecture; validation does not select an extra duration penalty |

<div class="caution-box"><strong>Important qualification.</strong> The weighting experiment succeeds in q2: validation selects the maximum tested duration weight of 8. In q4, however, it selects $w_d=1$. The data therefore do not support forcing stronger duration shrinkage for every regime count. Group weights should remain a validated hyperparameter and robustness device, not a predetermined economic conclusion.</div>


# 7. Preliminary Out-of-Sample Transition Results

The frozen models predict one-step regime probabilities from 3 January 2017 through 5 December 2023. This is a transition-probability backtest, not yet the full HMM-WGAN return and risk backtest planned for M5.

| Regimes | Model | Log loss ↓ | Brier score ↓ | Accuracy ↑ |
|---:|---|---:|---:|---:|
| 2 | Fixed transition | 0.191151 | 0.089790 | **0.952601** |
| 2 | MLR | 0.192744 | 0.090028 | **0.952601** |
| 2 | MLG | 0.172401 | 0.089265 | 0.952023 |
| 2 | Group-weighted MLG | **0.169999** | **0.087297** | 0.952023 |
| 4 | Fixed transition | 0.439116 | 0.208270 | **0.885549** |
| 4 | MLR | 0.460659 | 0.216206 | 0.880347 |
| 4 | MLG | **0.407523** | **0.204979** | 0.877457 |
| 4 | Group-weighted MLG | 0.408086 | 0.205294 | 0.876879 |

<table><tr>
<td width="50%"><img src="phase_1/transition_comparison/transition_selected_oos_metrics_q2.png" alt="Out-of-sample q2 transition-model metrics" width="100%"></td>
<td width="50%"><img src="phase_1/transition_comparison/transition_selected_oos_metrics_q4.png" alt="Out-of-sample q4 transition-model metrics" width="100%"></td>
</tr></table>
<p class="caption"><strong>Figure 3.</strong> Out-of-sample transition scores for q2 and q4. Lower log loss and Brier score are better. Source: Phase 1 transition comparison.</p>

**Inference.** Nonlinearity adds more value than linear covariate adjustment: original MLR is worse than the fixed matrix for both regime counts, while both MLG variants improve probability scores. The weighted MLG is best for q2, whereas the original MLG is marginally best for q4. The small reduction in MLG accuracy is not contradictory. Accuracy selects only the largest probability and rewards the dominant stay outcome; log loss and Brier evaluate the full vector and are more informative for simulation.

<img src="phase_1/transition_comparison/transition_matrix_comparison_mlg_q4.png" alt="Comparison of fixed and q4 MLG transition behavior" width="850">
<p class="caption"><strong>Figure 4.</strong> Fixed and MLG-implied q4 transition behavior. Source: Phase 1 transition comparison.</p>

These gains are promising but not sufficient to establish better portfolio risk forecasts. Experiment must test whether better one-step regime probabilities translate into more realistic regime paths, joint returns, tail losses, and exceedance sequences.


# 8. Feature Importance and the Duration-Hazard Result

## 8.1 Why importance must be measured two ways

Coefficient-group importance summarizes the norm of fitted state, market, and duration parameters. It is useful for comparing where the model allocates signal, but spline basis scaling and regularization affect magnitude, so it is not causal evidence. Conditional permutation provides a more outcome-oriented check: shuffle one feature group within current-state strata, recompute log loss, and ask how much predictive performance deteriorates. Stratification preserves the dominant state composition while disrupting incremental information.

<table><tr>
<td width="33%"><img src="phase_1/transition_comparison/feature_importance_mlr_q4.png" alt="q4 MLR feature importance" width="100%"></td>
<td width="33%"><img src="phase_1/transition_comparison/feature_importance_mlg_q4.png" alt="q4 original MLG feature importance" width="100%"></td>
<td width="33%"><img src="phase_1/transition_comparison/feature_importance_weighted_mlg_q4.png" alt="q4 group-weighted MLG feature importance" width="100%"></td>
</tr></table>
<p class="caption"><strong>Figure 5.</strong> q4 coefficient-based feature importance for original MLR, original MLG, and group-weighted MLG. Source: Phase 1 transition comparison.</p>

| Model and count | State importance | Market importance | Duration importance |
|---|---:|---:|---:|
| Original MLG, q2 | 10.43% | 28.34% | 61.22% |
| Group-weighted MLG, q2 | 24.64% | 56.82% | 18.54% |
| Original MLG, q4 | 13.04% | 20.61% | 66.35% |
| Group-weighted MLG, q4 | 11.76% | 28.40% | 59.84% |

For q2, the selected weight scheme changes the allocation substantially: duration falls from 61.22% to 18.54%, while market information rises to 56.82%, and out-of-sample log loss improves. For q4 the reallocation is smaller because validation selects no extra duration penalty.

<img src="phase_1/transition_feature_ablation/conditional_permutation_log_loss_q4.png" alt="q4 conditional-permutation log-loss changes by feature group" width="850">
<p class="caption"><strong>Figure 6.</strong> q4 conditional-permutation increase in log loss with uncertainty intervals. Positive values indicate useful incremental information. Source: Phase 1 feature-ablation analysis.</p>

Permutation evidence supports, rather than merely reflects, duration importance. For original MLG, shuffling duration increases prediction log loss by 0.0348 on average, with 95% interval [0.0188, 0.0536] and p=0.005. Shuffling the market group changes it by only 0.0006, interval [-0.0011, 0.0024], p=0.245. Thus duration contributes robust conditional predictive information in q4, while the selected current market set adds limited incremental test-period information after state and duration are known.

<div class="caution-box"><strong>Interpretation, not causation.</strong> A duration variable can proxy omitted market state, state-classification persistence, or a genuine semi-Markov hazard. The result says duration is predictively informative conditional on the implemented variables; it does not establish that elapsed time causes regime change.</div>


## 8.2 Duration hazard and MLG partial effects

A homogeneous HMM implies geometric durations and a constant exit hazard. A simple monotone-aging sensitivity model can instead write the stay odds as

$$\log\left(\frac{P(S_{t+1}=i\mid S_t=i,D_t=d)}{1-P(S_{t+1}=i\mid S_t=i,D_t=d)}\right)=\log\left(\frac{a_{ii}}{1-a_{ii}}\right)-\beta_i\log(d),\qquad \beta_i\ge 0.$$

Positive $\beta_i$ imposes aging: the longer a state lasts, the lower its stay odds. The preliminary constrained fit places all $\beta_i$ at zero for q2 and q4; unconstrained estimates are negative. Thus the data do **not** support a universal monotone aging rule. If anything, they indicate persistence in some states. This is consistent with the mixed geometric-duration tests and is why the main MLG estimates unconstrained state-specific nonlinear curves rather than hard-coding an increasing exit hazard.

<img src="phase_1/transition_comparison/duration_stay_probability_q4.png" alt="q4 stay probability by regime duration" width="850">
<p class="caption"><strong>Figure 7.</strong> Duration-conditioned q4 stay probabilities. The figure diagnoses persistence versus aging without assuming all regimes share one hazard shape. Source: Phase 1 transition comparison.</p>

A partial-effect curve holds other predictors at reference values and varies one duration term. For origin state 2, the figure below displays all economically relevant contrasts: staying in 2 versus moving from 2 to states 0, 1, or 3. Observed estimates use common duration bins so all curves are compared over identical observations. Jeffreys-binomial 95% intervals remain finite when a transition count is zero and expose sparse-support uncertainty.

<img src="phase_1/transition_comparison/mlg_partial_effects_q4/DURATION_STATE_2__all_contrasts.png" alt="q4 MLG duration-state-2 partial effects and observed log odds" width="900">
<p class="caption"><strong>Figure 8.</strong> Original q4 MLG partial effects and observed binned stay-versus-transition log odds for duration in state 2. Source: Phase 1 MLG partial effects.</p>

The early observed log odds are already high and then remain comparatively stable, whereas the fitted curve rises sharply from a lower initial value before flattening. The first common bin (durations 1–2) contains 104 observations: 92 stays, 12 transitions to state 0, and no transitions to states 1 or 3. For the rare 2-versus-1 contrast, the Jeffreys-adjusted observed log odds are approximately 5.22, but the 95% interval is very wide, approximately [3.59, 12.14]. Across all state-2 observations there are 771 stays, 48 moves to state 0, only 2 to state 1, and 4 to state 3.

The two panels below make clear that this evidence is not uniform across regimes. State 0 is the long benign regime: its direct transitions to states 1 and 3 are absent in the displayed bins, so their large log-odds and broad intervals should be read as sparse-data diagnostics, not as precise behavioral estimates. State 3 is a short stressed regime with much less duration support; its observed transition counts show why the fitted curve should be interpreted as regularized smoothing rather than a literal empirical hazard.

<table><tr>
<td width="50%"><img src="phase_1/transition_comparison/mlg_partial_effects_q4/DURATION_STATE_0__all_contrasts.png" alt="q4 MLG duration-state-0 partial effects and observed log odds" width="100%"></td>
<td width="50%"><img src="phase_1/transition_comparison/mlg_partial_effects_q4/DURATION_STATE_3__all_contrasts.png" alt="q4 MLG duration-state-3 partial effects and observed log odds" width="100%"></td>
</tr></table>
<p class="caption"><strong>Figure 9.</strong> Original q4 MLG duration partial effects for the benign state 0 and stressed state 3. Count heatmaps expose the support behind each observed log-odds estimate. Source: Phase 1 MLG partial effects.</p>

**Inference.** The broad direction is persistence rather than simple aging, but detailed curvature at short durations is uncertain and can be influenced by spline shrinkage and reference-covariate conditioning. Trying different spline degrees may be a useful sensitivity check; it should not be chosen by visual closeness to noisy bin means. M5 should compare degree 1, 2, and 3 using the same training-only significance and validation scoring protocol, with a minimum-support rule for transition-specific plots.


# 9. Pain Points and Project Responses

## Pain point 1: fixed transitions ignore market context and regime age

A fixed HMM matrix assumes that the same current regime always implies the same next-state probabilities. That is difficult to reconcile with changing credit, equity, commodity, and bond conditions and with evidence that some duration distributions are non-geometric. **Response:** estimate conditional transition probabilities from current/lagged states, market indices, and state-specific durations, while preserving the fixed matrix as the control model.

## Pain point 2: duration can dominate the fitted transition function

Original MLG coefficient norms allocate 61%–66% of importance to duration, and conditional permutation confirms that duration is genuinely predictive in q4. Nevertheless, dominance can crowd out weak but economically meaningful market signals or amplify a hard-state-label persistence mechanism. **Response:** the group-weighted MLG searches a lower market penalty and up to eight times stronger duration penalty. The response is evidence-based: q2 selects strong duration shrinkage and improves log loss, whereas q4 rejects extra duration shrinkage. We will report both models instead of forcing one weight scheme across counts.

## Pain point 3: transition log odds need not be linear

MLR assumes one constant slope per continuous predictor and destination contrast. Duration hazards can flatten, turn, or differ by current state, and market thresholds may matter more than average changes. **Response:** regularized logistic GAMs estimate smooth, predictor-specific effects, with roughness penalties, ridge stabilization, 5% significance screening, and untouched validation selection. The MLG probability-score gains over both fixed transition and MLR are preliminary evidence that the added flexibility is useful.

## Pain point 4: inferred regimes are uncertain labels, not directly observed facts

Transition models currently learn from decoded HMM states. q4’s mean maximum posterior probability is 0.8919, and 21.89% of observations have maximum posterior below 0.8, so hard labels discard meaningful ambiguity. This can exaggerate duration persistence and inject classification noise into rare transitions. **Response:** report posterior uncertainty, compare hard and soft transition targets, and evaluate whether posterior-weighted training improves calibration before changing the production architecture. The q4 sensitivity analysis already indicates that soft fixed-state probabilities can materially improve probability scores.

<img src="phase_1/hmm_transition_dynamics/transition_calibration_q4.png" alt="q4 hard-label and posterior-aware transition calibration" width="850">
<p class="caption"><strong>Figure 10.</strong> q4 transition calibration under hard and posterior-aware state treatments. Source: Phase 1 transition-dynamics analysis.</p>

## Pain point 5: rare transitions limit inference exactly where tail risk matters

Persistent regimes generate many stay observations but few transitions between specific stressed and benign states. This makes p-values, spline curvature, and empirical log-odds intervals unstable even in a long sample. **Response:** use regularization and bootstrap inference, display bin counts and uncertainty bands, pool evidence only when economically defensible, and require minimum transition support. In the WGAN stage, rare-state sample scarcity will also be addressed through balanced training diagnostics and sensitivity to regime count.


# 10. Possible Obstacles and Mitigation

| Obstacle | Consequence | Planned mitigation and decision point |
|---|---|---|
| Some market indices after 5 Dec. 2023 are not readily obtainable without paid access. | A longer backtest could use inconsistent coverage or introduce survivorship/proxy mismatch. | In M5, build a documented source audit and evaluate liquid, publicly available proxies. Re-estimate only after overlap, transformation, and correlation stability are verified; otherwise retain 2023 as the transparent common endpoint. |
| A full rolling HMM-WGAN run is computationally expensive, especially repeated WGAN training and Monte Carlo simulation. | Hyperparameter searches and uncertainty experiments may become infeasible or irreproducible. | Cache immutable Phase 1 features and selected models, vectorize transition sampling, parallelize independent regime generators, use deterministic seeds, and apply a staged search: small pilot, convergence check, then full simulation. Report wall time and hardware. |
| Regime instability may make one global transition model stale. | Backtest gains can be period-specific and disappear after structural breaks. | Use rolling or expanding windows, compare pre/post-stress performance, and retain q3/q4 sensitivity. Escalate to time-varying coefficients only if simpler rolling re-estimation is inadequate. |
| Rare stressed-state observations may be insufficient for stable WGAN training and tail tests. | Generator collapse or overly smooth tails could produce misleading VaR/CVaR. | Monitor regime sample counts, WGAN losses and generated moments; use balanced minibatch design where appropriate; report uncertainty across seeds; reduce regime count if a generator lacks minimum effective support. |
| Multiple model and diagnostic comparisons can encourage post-selection storytelling. | Reported improvements may be optimistic. | Freeze primary metrics and comparisons before M5, keep the 2017–2023 prediction sample untouched for Phase 1 selection, and distinguish confirmatory tests from exploratory sensitivity analyses. |

<div class="key-box"><strong>Governance rule.</strong> An extension will enter the primary model only if it improves prespecified validation criteria and remains interpretable and numerically stable. Otherwise it will be reported as a sensitivity analysis. This protects the project from complexity without measurable value.</div>


# 11. Milestone Plan

| Milestone | Scope | Main outputs | Completion criterion |
|---|---|---|---|
| **M4: Phase 1** | HMM calibration; q2/q3/q4 diagnostics; fixed, MLR, original MLG, and weighted MLG selection; transition probability backtest; feature and partial-effect interpretation | Reproducible CSVs/plots, this proposal, selected model specifications, limitations log | Diagnostics explained; selection uses training/validation only; 2017–2023 comparison reproduced from stored probabilities |
| **M5: Full model** | Run fixed-transition and dynamic-transition HMM-WGAN versions; address post-2023 data proxies; test spline-degree and posterior-label sensitivities where material | Simulated regime paths and returns; stylized-fact, dependence, VaR/CVaR, exceedance, and joint VaR–ES comparisons | Same windows/seeds/portfolios across variants; computational and data-proxy decisions documented |
| **M6: Draft report** | Integrate methodology, diagnostics, simulation, backtests, sensitivity analyses, and limitations | Complete draft report plus appendix and reproducibility checklist | Every research question mapped to evidence; tables and figures independently regenerated |
| **M7: Final report** | Resolve feedback, finalize inference, and package code and outputs | Final report, presentation-ready figures, clean repository instructions | Conclusions bounded by evidence; repository reproduces principal tables and plots end to end |

## Immediate M5 work packages

1. Freeze q4 primary and q3 robustness HMM configurations.
2. Audit post-2023 proxy candidates and define a common data endpoint.
3. Run controlled spline-degree sensitivity without using the test set for selection.
4. Calibrate regime-specific WGANs with repeated seeds and convergence diagnostics.
5. Generate matched fixed and dynamic regime paths and multivariate returns.
6. Evaluate stylized facts, cross-asset dependence, VaR/CVaR, Christoffersen coverage/independence, and joint VaR–ES loss (Christoffersen; Fissler and Ziegel).


# 12. Expected Contribution and Success Criteria

The project will contribute an interpretable bridge between regime detection and flexible multivariate generation. It is successful only if the dynamic transition layer adds measurable value beyond a fixed HMM-WGAN, not merely because it is more complex.

**Primary empirical success criteria**

1. Dynamic transitions improve out-of-sample log loss and Brier score relative to the fixed matrix under q4, with q3 reported as robustness.
2. Dynamic HMM-WGAN simulations better reproduce prespecified marginal and multivariate stylized facts without degrading stability across seeds.
3. Portfolio VaR and CVaR estimates improve coverage, independence, and severity-sensitive loss relative to the fixed-transition benchmark.
4. Feature and partial-effect conclusions remain directionally stable across reasonable knot, spline-degree, state-label, and window sensitivities.
5. Runtime and data requirements are documented well enough for a third party to reproduce the principal results.

**Current preliminary inference.** Phase 1 meets the first criterion for the nonlinear models: original q4 MLG improves fixed-transition log loss by approximately 7.2% and Brier score by approximately 1.6%, while linear MLR does not improve the benchmark. Duration is an important predictor, but its effect is persistence-oriented and heterogeneous rather than a universal aging law. Four regimes remain defensible but not unequivocal because q3 generalizes better under Gaussian HMM likelihood. These findings justify proceeding to the full model while fixing clear robustness checks in advance.

<div class="key-box"><strong>Proposed answer to the research question at M4.</strong> Dynamic nonlinear transition probabilities are more accurate than fixed transitions for the observed regime sequence, but the evidence does not yet establish superior return simulation or risk measurement. M5 is the necessary causal comparison at the system level.</div>


# Works Cited

Ang, Andrew, and Allan Timmermann. “Regime Changes and Financial Markets.” *Annual Review of Financial Economics*, vol. 4, 2012, pp. 313–337.

Arjovsky, Martin, Soumith Chintala, and Léon Bottou. “Wasserstein Generative Adversarial Networks.” *Proceedings of the 34th International Conference on Machine Learning*, vol. 70, 2017, pp. 214–223.

Biernacki, Christophe, Gilles Celeux, and Gérard Govaert. “Assessing a Mixture Model for Clustering with the Integrated Completed Likelihood.” *IEEE Transactions on Pattern Analysis and Machine Intelligence*, vol. 22, no. 7, 2000, pp. 719–725. doi:10.1109/34.865189.

Bollerslev, Tim. “Generalized Autoregressive Conditional Heteroskedasticity.” *Journal of Econometrics*, vol. 31, no. 3, 1986, pp. 307–327.

Brier, Glenn W. “Verification of Forecasts Expressed in Terms of Probability.” *Monthly Weather Review*, vol. 78, no. 1, 1950, pp. 1–3.

Christoffersen, Peter F. “Evaluating Interval Forecasts.” *International Economic Review*, vol. 39, no. 4, 1998, pp. 841–862.

Cont, Rama. “Empirical Properties of Asset Returns: Stylized Facts and Statistical Issues.” *Quantitative Finance*, vol. 1, no. 2, 2001, pp. 223–236.

Durland, J. Michael, and Thomas H. McCurdy. “Duration-Dependent Transitions in a Markov Model of U.S. GNP Growth.” *Journal of Business & Economic Statistics*, vol. 12, no. 3, 1994, pp. 279–288.

Eilers, Paul H. C., and Brian D. Marx. “Flexible Smoothing with B-Splines and Penalties.” *Statistical Science*, vol. 11, no. 2, 1996, pp. 89–121.

Engle, Robert F. “Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation.” *Econometrica*, vol. 50, no. 4, 1982, pp. 987–1007.

Engle, Robert. “Dynamic Conditional Correlation: A Simple Class of Multivariate Generalized Autoregressive Conditional Heteroskedasticity Models.” *Journal of Business & Economic Statistics*, vol. 20, no. 3, 2002, pp. 339–350.

Filardo, Andrew J. “Business-Cycle Phases and Their Transitional Dynamics.” *Journal of Business & Economic Statistics*, vol. 12, no. 3, 1994, pp. 299–308.

Fissler, Tobias, and Johanna F. Ziegel. “Higher Order Elicitability and Osband’s Principle.” *The Annals of Statistics*, vol. 44, no. 4, 2016, pp. 1680–1707.

Giner, Javier, and Valeriy Zakamulin. “The State-Dependent Duration Model: An Alternative to Hidden Markov Models.” *Economic Modelling*, vol. 122, 2023, article 106237.

Goodfellow, Ian, et al. “Generative Adversarial Nets.” *Advances in Neural Information Processing Systems 27*, 2014, pp. 2672–2680.

Guidolin, Massimo. “Markov Switching Models in Empirical Finance.” *Advances in Econometrics*, vol. 27B, 2011, pp. 1–86.

Gulrajani, Ishaan, et al. “Improved Training of Wasserstein GANs.” *Advances in Neural Information Processing Systems 30*, 2017.

Hamilton, James D. “A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle.” *Econometrica*, vol. 57, no. 2, 1989, pp. 357–384.

Hastie, Trevor, and Robert Tibshirani. *Generalized Additive Models*. Chapman and Hall, 1990.

Istiaque, Asif, Chi Seng Pun, and Haoran Yong. “HMM-WGAN: A Novel Model for Time Series Simulation.” *Quantitative Finance*, vol. 25, no. 6, 2025, pp. 873–893. doi:10.1080/14697688.2025.2511115.

Jarque, Carlos M., and Anil K. Bera. “A Test for Normality of Observations and Regression Residuals.” *International Statistical Review*, vol. 55, no. 2, 1987, pp. 163–172.

Ljung, Greta M., and George E. P. Box. “On a Measure of Lack of Fit in Time Series Models.” *Biometrika*, vol. 65, no. 2, 1978, pp. 297–303.

Maheu, John M., and Thomas H. McCurdy. “Identifying Bull and Bear Markets in Stock Returns.” *Journal of Business & Economic Statistics*, vol. 18, no. 1, 2000, pp. 100–112.

Masi, Giovanni, et al. “Time-Series Generative Adversarial Networks for Financial Market Simulation.” *Proceedings of the Fourth ACM International Conference on AI in Finance*, 2023, pp. 524–532.

McLachlan, Geoffrey J. “On Bootstrapping the Likelihood Ratio Test Statistic for the Number of Components in a Normal Mixture.” *Applied Statistics*, vol. 36, no. 3, 1987, pp. 318–324.

Nguyen, Truong Co, Nam Hoang, and Trung Hieu Ha. “Dynamic Duration-Dependent HMM-WGAN for Multivariate Financial Return Simulation and Risk Measurement: Literature Review and Competitor Analysis.” WorldQuant University, 5 July 2026.

Nguyen, Truong Co, Nam Hoang, and Trung Hieu Ha. “Dynamic Duration-Dependent HMM-WGAN for Multivariate Financial Return Simulation and Risk Measurement: Problem Statement.” WorldQuant University, 27 June 2026.

Rabiner, Lawrence R. “A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition.” *Proceedings of the IEEE*, vol. 77, no. 2, 1989, pp. 257–286.

Schwarz, Gideon. “Estimating the Dimension of a Model.” *The Annals of Statistics*, vol. 6, no. 2, 1978, pp. 461–464. doi:10.1214/aos/1176344136.

Takahashi, Shuntaro, Yu Chen, and Kumiko Tanaka-Ishii. “Modeling Financial Time-Series with Generative Adversarial Networks.” *Physica A*, vol. 527, 2019, article 121261.

Wiese, Magnus, et al. “Quant GANs: Deep Generation of Financial Time Series.” *Quantitative Finance*, vol. 20, no. 9, 2020, pp. 1419–1440.

Yu, Shun-Zheng. *Hidden Semi-Markov Models: Theory, Algorithms and Applications*. Elsevier, 2015.
